<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 7: Hiperparametre Optimizasyonu

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 7 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta07/hafta07_hiperparametre_avi.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta07/hafta07_hiperparametre_avi.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta07_ensemble_modeller.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/07/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - GridSearchCV ve RandomizedSearchCV
> - Optimal parametre arama
> - Overfitting ve Regularization

# Hafta 7 — Hiperparametre Ayarlama (Tuning)

Bu defterde makine öğrenmesi modellerinin hiperparametrelerini nasıl optimize edeceğimizi öğreneceğiz.

## İçindekiler
1. Hiperparametre Nedir?
2. GridSearchCV ile Arama
3. RandomizedSearchCV ile Arama
4. En İyi Parametreler ve Skor
5. Öncesi/Sonrası Karşılaştırma
6. Öğrenme Eğrisi (Learning Curve)

## 1. Hiperparametre Nedir?

Makine öğrenmesinde iki tür parametre vardır:

### Model Parametreleri (Öğrenilen)
- Model eğitimi sırasında veriden **otomatik olarak öğrenilir**
- Örnek: Lojistik regresyon katsayıları, ağaçtaki bölme noktaları

### Hiperparametreler (Ayarlanan)
- Eğitim **öncesinde kullanıcı tarafından belirlenir**
- Modelin nasıl öğreneceğini kontrol eder
- Doğru hiperparametreler modelin performansını önemli ölçüde artırabilir

### Random Forest Hiperparametreleri

| Hiperparametre | Açıklama | Tipik Aralık |
|----------------|----------|-------------|
| `n_estimators` | Ağaç sayısı | 50–500 |
| `max_depth` | Maksimum ağaç derinliği | 3–20, None |
| `min_samples_split` | Bölme için minimum örnek sayısı | 2–20 |
| `min_samples_leaf` | Yaprakta minimum örnek sayısı | 1–10 |
| `max_features` | Her bölmede dikkate alınan özellik sayısı | 'sqrt', 'log2', None |

### Arama Yöntemleri
- **GridSearchCV:** Tüm kombinasyonları dener — kapsamlı ama yavaş
- **RandomizedSearchCV:** Rastgele kombinasyonları dener — hızlı ve verimli

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    learning_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint, uniform

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Kütüphaneler yüklendi!")

### Eğitim ve Test Setlerine Ayırma

Veriyi eğitim ve test olarak ikiye bölüyoruz. `stratify` parametresi, her iki sette de sınıf dağılımının aynı kalmasını sağlar. `random_state` ile tekrarlanabilir sonuçlar elde ediyoruz.

In [ ]:
# Wine veri seti
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Veri seti boyutu: {X.shape}")
print(f"Sınıf sayısı: {len(np.unique(y))}")
print(f"Eğitim: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 2. GridSearchCV ile Hiperparametre Arama

**GridSearchCV**, tanımlanan parametre ızgarasındaki **tüm kombinasyonları** dener ve en iyi skoru veren parametreleri bulur.

In [ ]:
# Varsayılan model (karşılaştırma için)
default_rf = RandomForestClassifier(random_state=42)
default_rf.fit(X_train, y_train)
default_pred = default_rf.predict(X_test)
default_acc = accuracy_score(y_test, default_pred)
print(f"Varsayılan Random Forest Doğruluğu: {default_acc:.4f}")
print(f"Varsayılan parametreler:")
print(f"  n_estimators:     {default_rf.n_estimators}")
print(f"  max_depth:        {default_rf.max_depth}")
print(f"  min_samples_split: {default_rf.min_samples_split}")

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# GridSearchCV parametre ızgarası
param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

# Toplam kombinasyon sayısı
total = 1
for v in param_grid.values():
    total *= len(v)
print(f"Toplam aranacak kombinasyon: {total}")

# GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nEn iyi parametreler: {grid_search.best_params_}")
print(f"En iyi çapraz doğrulama skoru: {grid_search.best_score_:.4f}")

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# GridSearchCV sonuçlarını görselleştir
results = pd.DataFrame(grid_search.cv_results_)

# n_estimators vs max_depth ısı haritası (min_samples_split=en iyi değer)
best_mss = grid_search.best_params_['min_samples_split']
mask = results['param_min_samples_split'] == best_mss
subset = results[mask]

pivot = subset.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators'
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd')
plt.title(f'GridSearchCV Sonuçları\n(min_samples_split={best_mss})')
plt.xlabel('n_estimators')
plt.ylabel('max_depth')
plt.tight_layout()
plt.show()

## 3. RandomizedSearchCV ile Hiperparametre Arama

**RandomizedSearchCV**, parametre uzayından **rastgele örnekler** çeker. Daha geniş bir aralık tanımlanabilir ve daha hızlı sonuç verir.

In [ ]:
# RandomizedSearchCV parametre dağılımları
param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=50,  # 50 rastgele kombinasyon dene
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f"\nEn iyi parametreler: {random_search.best_params_}")
print(f"En iyi çapraz doğrulama skoru: {random_search.best_score_:.4f}")

## 4. En İyi Parametreler ve Skor

### Model Değerlendirme

Modelin performansını çeşitli metriklerle değerlendiriyoruz: doğruluk (accuracy), kesinlik (precision), duyarlılık (recall) ve F1-skoru.

In [ ]:
# İki arama yöntemini karşılaştır
print("="*60)
print("GridSearchCV vs RandomizedSearchCV Karşılaştırması")
print("="*60)

print(f"\nGridSearchCV:")
print(f"  En iyi parametreler: {grid_search.best_params_}")
print(f"  En iyi CV skoru:     {grid_search.best_score_:.4f}")
grid_test_acc = accuracy_score(y_test, grid_search.predict(X_test))
print(f"  Test doğruluğu:      {grid_test_acc:.4f}")

print(f"\nRandomizedSearchCV:")
print(f"  En iyi parametreler: {random_search.best_params_}")
print(f"  En iyi CV skoru:     {random_search.best_score_:.4f}")
random_test_acc = accuracy_score(y_test, random_search.predict(X_test))
print(f"  Test doğruluğu:      {random_test_acc:.4f}")

## 5. Öncesi/Sonrası Karşılaştırma

### Hiperparametre Optimizasyonu

Modelin en iyi ayarlarını bulmak için farklı parametre kombinasyonlarını sistematik olarak deniyoruz. Bu süreç model performansını önemli ölçüde artırabilir.

In [ ]:
# Öncesi ve sonrası
comparison = pd.DataFrame({
    'Model': ['Varsayılan RF', 'GridSearchCV RF', 'RandomizedSearchCV RF'],
    'Test Doğruluğu': [default_acc, grid_test_acc, random_test_acc]
})

print("\nModel Karşılaştırması")
print("=" * 50)
print(comparison.to_string(index=False))

# İyileşme oranı
best_tuned = max(grid_test_acc, random_test_acc)
improvement = best_tuned - default_acc
print(f"\nİyileşme: {improvement:+.4f} ({improvement/default_acc*100:+.2f}%)")

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Karşılaştırma çubuk grafiği
plt.figure(figsize=(10, 6))
colors = ['#95a5a6', '#2196F3', '#4CAF50']
bars = plt.bar(comparison['Model'], comparison['Test Doğruluğu'], 
               color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, comparison['Test Doğruluğu']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', fontsize=13, fontweight='bold')

plt.ylabel('Test Doğruluğu')
plt.title('Hiperparametre Ayarlama — Öncesi vs Sonrası')
plt.ylim(0.85, 1.02)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Model Değerlendirme

Modelin performansını çeşitli metriklerle değerlendiriyoruz: doğruluk (accuracy), kesinlik (precision), duyarlılık (recall) ve F1-skoru.

In [ ]:
# En iyi modelin sınıflandırma raporu
best_model = grid_search.best_estimator_ if grid_test_acc >= random_test_acc else random_search.best_estimator_
best_name = 'GridSearchCV' if grid_test_acc >= random_test_acc else 'RandomizedSearchCV'

print(f"En İyi Model: {best_name}")
print(f"\nSınıflandırma Raporu:")
print(classification_report(y_test, best_model.predict(X_test), 
                            target_names=wine.target_names))

## 6. Öğrenme Eğrisi (Learning Curve)

**Öğrenme eğrisi**, eğitim seti büyüklüğüne göre modelin performansını gösterir. Aşırı öğrenme (overfitting) ve eksik öğrenme (underfitting) durumlarını tespit etmemize yardımcı olur.

In [ ]:
# Varsayılan ve en iyi model için öğrenme eğrileri
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_lc = {
    'Varsayılan RF': RandomForestClassifier(random_state=42),
    f'Optimize Edilmiş RF ({best_name})': best_model
}

for ax, (name, model) in zip(axes, models_lc.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy'
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                    alpha=0.15, color='#2196F3')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                    alpha=0.15, color='#FF9800')
    ax.plot(train_sizes, train_mean, 'o-', color='#2196F3', linewidth=2, 
            label='Eğitim Skoru')
    ax.plot(train_sizes, val_mean, 'o-', color='#FF9800', linewidth=2, 
            label='Doğrulama Skoru')
    
    ax.set_xlabel('Eğitim Seti Boyutu')
    ax.set_ylabel('Doğruluk')
    ax.set_title(f'Öğrenme Eğrisi\n{name}')
    ax.legend(loc='lower right')
    ax.set_ylim(0.7, 1.05)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Öğrenme Eğrisini Yorumlama

| Durum | Eğitim Eğrisi | Doğrulama Eğrisi | Yorum |
|-------|---------------|-------------------|--------|
| **Aşırı Öğrenme** | Çok yüksek | Düşük | Eğitim ve doğrulama arasında büyük fark |
| **Eksik Öğrenme** | Düşük | Düşük | Her iki eğri de düşük |
| **İyi Uyum** | Yüksek | Yüksek (yakın) | Her iki eğri yakınsıyor |

## Özet

Bu defterde öğrendiklerimiz:

1. **Hiperparametreler**, model eğitimi öncesinde ayarlanan ve modelin öğrenme davranışını belirleyen değerlerdir
2. **GridSearchCV** tüm kombinasyonları deneyerek en iyisini bulur — kesin ama yavaş
3. **RandomizedSearchCV** rastgele örnekleme ile arar — hızlı ve genellikle yeterince iyi
4. Hiperparametre optimizasyonu modelin performansını **gözle görülür şekilde** iyileştirebilir
5. **Öğrenme eğrileri**, modelin aşırı mı yoksa eksik mi öğrendiğini anlamamıza yardımcı olur

### Pratik Tavsiyeler
- Önce **RandomizedSearchCV** ile geniş bir aralıkta arayın
- Sonra en iyi bölge etrafında **GridSearchCV** ile ince ayar yapın
- Her zaman **çapraz doğrulama** kullanın — tek bir test setine güvenmeyin
- Aşırı iyimser sonuçlara dikkat edin — eğitim ve test performansı arasındaki farkı kontrol edin

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>